# Lab 14: People Tracking and In/Out Counting

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Lab 16 the program counted the objects in **one picture** and forgot them in the next. A shop, a library or a lift needs more: how many people **came in** and how many **went out**? To answer, the program must know that the person in this picture is the **same person** as in the last picture. This is called **tracking**. In this lab you give every person a number (an ID), follow the person across the picture and count each time the person crosses a **virtual line**. The count is saved in a log that opens in Excel.

**By the end of this lab you can:**
1. Explain the difference between detection and tracking, and why a tracker gives every person an ID.
2. Run a tracker on the live camera and read the ID and the trail of each person.
3. Count people going in and out with a virtual line, and calculate how many are inside.
4. Explain why the program ignores a small zone around the line (the margin).
5. Test the counter and measure how accurate it is.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Privacy.** The program follows people, so use it only on yourselves and on classmates who agree. It stores numbers, not faces, but the alert pictures of other labs could show faces. Never use it on people who do not know about it.

## 2. Background

**Detection and tracking.** A detector (the model of Lab 10) looks at **one picture** and draws boxes. It has no memory. In the next picture it draws new boxes and does not know which box is which person. A **tracker** adds the memory. It compares the boxes of this picture with the boxes of the last picture and gives the same **ID number** to boxes that match.

| | Detector | Tracker |
|---|---|---|
| Input | One picture | A stream of pictures |
| Output | Boxes and classes | Boxes, classes and an **ID** for each box |
| Memory | None | Remembers each person for a while |
| Good for | "How many cups are in view now?" | "How many people walked in today?" |

This lab uses **ByteTrack**, a tracker that is built into Ultralytics. It matches boxes by how much they overlap with the boxes of the last picture. It needs no extra model, so the FPS stays close to the FPS of Lab 15.

**The virtual line.** To count people going in and out, draw a line across the picture. A person is **before** the line (side -1) or **after** it (side +1). When the side of the same ID changes, the person has **crossed**.

| Move of the person | Side changes | Counted as |
|---|---|---|
| left to right (or top to bottom) | -1 to +1 | IN |
| right to left (or bottom to top) | +1 to -1 | OUT |

**The margin.** A box jumps a few pixels from picture to picture. A person who stands on the line would flip from side to side and be counted many times. To prevent this, the program ignores a zone of `MARGIN` pixels on both sides of the line. The side only changes when the centre of the box is clearly on the other side.

**People inside.** If no one was inside at the start, then `inside = IN - OUT`. People who were already inside when you started the program are not counted, so the real number can be higher.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(10)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (20 min)

### Step 1: Get the Model Files
This is the same step as Lab 16. The fast model (`yolo26n_fp16.engine`) is the one you built in Lab 15. If it is missing, the program uses the slower `yolo26n.pt` and still works.

In [ ]:
import glob
import os
import shutil

print("This notebook is saved in:", os.getcwd())

for source in ["../lab15/yolo26n_fp16.engine", "../lab16/yolo26n_fp16.engine", "../lab15/yolo26n.pt", "../models/yolo26n.pt"]:
    name = os.path.basename(source)
    if os.path.exists(source) and not os.path.exists(name):
        shutil.copy(source, name)
        print("copied", source)

print("Model files here:", sorted(glob.glob("*.engine") + glob.glob("*.pt")))
if not os.path.exists("yolo26n_fp16.engine"):
    print("No FP16 engine here. The program will use yolo26n.pt (slower). Build the engine in Lab 15 to make it faster.")

### Step 2: Install the Tracker Part
ByteTrack needs a small package called `lap` (it solves the matching problem). The cell installs it only if it is missing. This needs the internet and takes about 10 seconds the first time.

In [ ]:
import subprocess
import sys

try:
    import lap
    print("lap is installed.")
except ImportError:
    print("Installing lap ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lap"], check=True)
    import lap
    print("lap is installed.")

### Step 3: Plan Your Count
Think about where the counter could be used. Where would the camera be, and where is the line? The people must cross the line **sideways** to the line (not walk along it).

| Place | Camera | Line | IN means |
|---|---|---|---|
| Classroom door | Above the door | Across the doorway | Entering the room |
| Shop entrance | On the wall, facing the door | Across the picture | Entering the shop |
| Corridor | At the end of the corridor | Across the corridor | Walking towards the camera |
| Your desk (for this lab) | On the desk, facing the room | Down the middle of the picture | Walking left to right |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Where would you use a people counter in real life?:
- Who would read the number, and what would they do with it?:
- Which side is IN in your set-up (left to right, or right to left)?:


---
# Part B: Track People (35 min)

### Step 1: The Settings
Words in capital letters are settings you can change. Run the cell.

In [ ]:
# ---------------- Settings you can change ----------------
LINE = "vertical"        # "vertical" (people walk left or right) or "horizontal" (people walk up or down)
FLIP = False             # True swaps IN and OUT
MARGIN = 12              # pixels around the line that are ignored
CONFIDENCE = 0.40        # confidence limit
ROOM_LIMIT = 3           # show ROOM FULL when this many people or more are inside
MODEL = "yolo26n_fp16.engine"
# ----------------------------------------------------------
print("Line:", LINE, "| margin:", MARGIN, "| room limit:", ROOM_LIMIT)

### Step 2: Load the Model
The first time the engine loads, it can take a few seconds.

In [ ]:
import csv
import time
from collections import deque
from datetime import datetime

import cv2
import torch
from ultralytics import YOLO

import labview

device = 0 if torch.cuda.is_available() else "cpu"
if not os.path.exists(MODEL):
    print(f"{MODEL} not found, using yolo26n.pt instead (slower)")
    MODEL = "yolo26n.pt"
model = YOLO(MODEL, task="detect")
print("Using model:", MODEL)

### Step 3: Tracking Only
First, track without counting. The function `draw_people` gives every person a box, an **ID** (`#1`, `#2`, ...) and a **trail** (the last 30 positions of the centre). The line `model.track(...)` is the only change from the `model(...)` call of Lab 16:

| Part | What it does |
|---|---|
| `persist=True` | Keep the memory from the last picture (without it every picture starts new and the IDs change all the time) |
| `tracker="bytetrack.yaml"` | Use the ByteTrack tracker |
| `classes=[0]` | Look at class 0 only. In the list of Lab 10, class 0 is `person`. |

In [ ]:
run = {"trails": {}, "last_side": {}, "n_in": 0, "n_out": 0, "note": "", "warned": False}


def reset_tracker():
    """Forget all people: the tracker's memory, the trails, the sides and the counts."""
    predictor = getattr(model, "predictor", None)
    for tracker in (getattr(predictor, "trackers", None) or []):
        tracker.reset()
    run.update(trails={}, last_side={}, n_in=0, n_out=0, note="")


def track_people(frame, confidence):
    """Run the tracker on one picture. Returns a list of (id, x1, y1, x2, y2)."""
    result = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=confidence,
                         classes=[0], device=device, verbose=False)[0]
    if result.boxes.id is None:                       # nobody is tracked in this picture
        return []
    boxes = result.boxes.xyxy.int().tolist()
    ids = result.boxes.id.int().tolist()
    return [(i, *b) for i, b in zip(ids, boxes)]


def draw_people(shown, people):
    """Draw the box, the ID and the trail of every person."""
    now = {p[0] for p in people}
    for old in [i for i in run["trails"] if i not in now]:
        del run["trails"][old]                         # forget the trail of people who left the picture
    for tid, x1, y1, x2, y2 in people:
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        trail = run["trails"].setdefault(tid, deque(maxlen=30))
        trail.append((cx, cy))
        cv2.rectangle(shown, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(shown, f"#{tid}", (x1, max(y1 - 8, 18)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        for a, b in zip(list(trail), list(trail)[1:]):
            cv2.line(shown, a, b, (255, 160, 0), 2)
        cv2.circle(shown, (cx, cy), 4, (0, 0, 255), -1)


def process_track(frame, state):
    shown = frame.copy()
    people = track_people(frame, state["Confidence"])
    draw_people(shown, people)
    cv2.putText(shown, f"People: {len(people)}   {state['fps']:.0f} FPS", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    return shown


print("The tracking program is ready.")

### Step 4: Run the Tracker
Run the cell. Walk slowly in front of the camera (one or two people). Look at the number above each box: it should **stay the same** while the person is in view. The orange line behind the person is the trail. Press **Stop** when you are done.

Try these three tests and write what you see in the answer cell:
1. One person walks from the left of the picture to the right.
2. Two people walk past each other.
3. One person walks behind a book or a chair for a second and comes back.

In [ ]:
reset_tracker()
live = labview.live(process_track, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE)})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen:
- Test 1: did the ID stay the same all the way across?:
- Test 2: what happened to the IDs when two people passed each other?:
- Test 3: what happened to the ID after the person was hidden for a second?:
- What is the ID of a person who goes out of the picture and comes back after 10 seconds?:


> **Why do IDs sometimes change?** The tracker only knows boxes. When two people overlap, or when a person is hidden for too long, the tracker can lose the person. When the person appears again, it is a new box without a match, so it gets a **new ID**. Tracking is good but not perfect, so a counter built on it is also not perfect. You will measure this in Part E.

---
# Part C: Count People In and Out (50 min)

### Step 1: The Counting Program
Read this cell before you run it. The new parts are:

| Function | What it does |
|---|---|
| `side_of(position, line, margin)` | Gives -1 (before the line), +1 (after the line) or 0 (inside the margin) |
| `check_crossing(tid, position, line)` | Remembers the last side of each ID. When the side changes, it returns `"IN"` or `"OUT"`, otherwise `None` |
| `log_event(tid, direction)` | Adds a line to `people_log.csv` and rewrites `people_log.xlsx` |
| `process(frame, state)` | Tracks, checks the crossing for each person, draws the line and the numbers. The live view calls it for every picture. |

Find the five steps of a vision application from Lab 16 in the code: capture (the live view), detect and track (`track_people`), decide (`check_crossing`), act (the drawing) and record (`log_event`).

In [ ]:
def side_of(position, line, margin):
    """-1 before the line, +1 after the line, 0 in the margin around the line."""
    if position < line - margin:
        return -1
    if position > line + margin:
        return 1
    return 0


def check_crossing(tid, position, line):
    """Return "IN", "OUT" or None for this person (this ID)."""
    side = side_of(position, line, MARGIN)
    if side == 0:
        return None                                    # in the margin: keep the last side, count nothing
    before = run["last_side"].get(tid)
    run["last_side"][tid] = side
    if before is None or before == side:
        return None                                    # first time we see this ID, or no change of side
    moved_forward = side > before                      # from -1 to +1
    return "IN" if moved_forward != FLIP else "OUT"


def write_excel():
    """Rewrite people_log.xlsx from people_log.csv, so it opens in Excel with a header and number columns."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("people_log.xlsx needs openpyxl. people_log.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "People"
    with open("people_log.csv", newline="") as f:
        rows = list(csv.reader(f))
    for i, row in enumerate(rows):
        if i > 0:
            row[1], row[3], row[4], row[5] = int(row[1]), int(row[3]), int(row[4]), int(row[5])
        sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCDEF", (20, 8, 11, 8, 8, 8)):
        sheet.column_dimensions[column].width = width
    book.save("people_log.xlsx")


def log_event(tid, direction):
    """Add one line to the log files (people_log.csv and people_log.xlsx)."""
    first = not os.path.exists("people_log.csv")
    inside = max(0, run["n_in"] - run["n_out"])
    with open("people_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "id", "direction", "in_total", "out_total", "inside"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), tid, direction, run["n_in"], run["n_out"], inside])
    write_excel()


def process(frame, state):
    shown = frame.copy()
    h, w = shown.shape[:2]
    vertical = LINE == "vertical"
    line = int((w if vertical else h) * state["Line position"])
    people = track_people(frame, state["Confidence"])
    draw_people(shown, people)

    for tid, x1, y1, x2, y2 in people:
        position = (x1 + x2) // 2 if vertical else (y1 + y2) // 2
        direction = check_crossing(tid, position, line)
        if direction == "IN":
            run["n_in"] += 1
        elif direction == "OUT":
            run["n_out"] += 1
        if direction:
            log_event(tid, direction)
            run["note"] = f"#{tid} went {direction} at {datetime.now().strftime('%H:%M:%S')}"

    # the virtual line and its margin
    if vertical:
        cv2.line(shown, (line, 0), (line, h), (0, 255, 255), 2)
        cv2.line(shown, (line - MARGIN, 0), (line - MARGIN, h), (0, 140, 140), 1)
        cv2.line(shown, (line + MARGIN, 0), (line + MARGIN, h), (0, 140, 140), 1)
    else:
        cv2.line(shown, (0, line), (w, line), (0, 255, 255), 2)
        cv2.line(shown, (0, line - MARGIN), (w, line - MARGIN), (0, 140, 140), 1)
        cv2.line(shown, (0, line + MARGIN), (w, line + MARGIN), (0, 140, 140), 1)

    inside = max(0, run["n_in"] - run["n_out"])
    full = inside >= int(state["Room limit"])
    colour = (0, 0, 255) if full else (0, 255, 0)
    cv2.putText(shown, f"IN {run['n_in']}   OUT {run['n_out']}   INSIDE {inside}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, colour, 2)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if full:
        cv2.putText(shown, "ROOM FULL", (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    if run["note"]:
        cv2.putText(shown, run["note"][:70], (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return shown


print("The counting program is ready.")

### Step 2: Run the Counter
Run the cell. A yellow line appears in the picture (the two thin lines are the margin). Walk across the line from the IN side to the OUT side and back, and watch `IN`, `OUT` and `INSIDE` at the top. Use the sliders to move the line and to change the confidence and the room limit while the program runs. Press **Stop** when you are done.

The first crossing writes the log files `people_log.csv` and `people_log.xlsx` in the folder of this notebook.

In [ ]:
reset_tracker()
live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE),
                                       "Line position": (0.1, 0.9, 0.05, 0.5),
                                       "Room limit": (1, 10, 1, ROOM_LIMIT)})

### Step 3: See the Results
Run this cell after you pressed Stop. It shows the log and a small summary. To open the Excel file, use the file browser on the left of JupyterLab, right-click `people_log.xlsx` and choose **Download**, then open it on your laptop.

In [ ]:
def show_results():
    for name in ("people_log.csv", "people_log.xlsx"):
        print(f"{name:<18}", "found" if os.path.exists(name) else "not created yet (it appears at the first crossing)")
    if os.path.exists("people_log.csv"):
        with open("people_log.csv", newline="") as f:
            rows = list(csv.DictReader(f))
        ins = sum(1 for r in rows if r["direction"] == "IN")
        outs = sum(1 for r in rows if r["direction"] == "OUT")
        print()
        print("Crossings in the log:", len(rows), "| IN:", ins, "| OUT:", outs)
        print()
        with open("people_log.csv") as f:
            print(f.read())


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- IN and OUT shown at the end of your test:
- Did the number of IN and OUT match the number of times you crossed the line?:
- First line of the log after the header:
- What happens when you stand on the line?:


---
# Part D: Make It Your Own (30 min)
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run Step 2 of Part C again.

- Switch to `LINE = "horizontal"` and walk towards and away from the camera. Which side is IN now?
- Use the **bottom centre** of the box (the feet) instead of the centre of the box. Change `cy = (y1 + y2) // 2` to `cy = y2` in `draw_people`, and `(y1 + y2) // 2` to `y2` in `process`. Why is this better for a camera that looks down from above?
- Show the time of the last crossing, or the busiest minute, on the screen.
- Save a picture each time someone goes IN (use `cv2.imwrite` as in Lab 16).
- Make the room limit change the colour of the line, too.
- Count only the people whose box is wider than 80 pixels (to ignore people far away).

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


---
# Part E: Test and Present (45 min)

### Step 1: Measure the Accuracy
A counter is only useful if it is right. Delete the old log, run the counter again and **cross the line exactly 10 times** in each situation below (5 IN and 5 OUT). Write the numbers the program shows. A test has more value when one person counts the real crossings aloud and another person watches the screen.

In [ ]:
# Delete the old log to start a clean test
for name in ("people_log.csv", "people_log.xlsx"):
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("people_log*")))

Run Step 2 of Part C for each test. Run the cell of Step 2 again before each test: it starts new counts.

| Test | Situation | Real IN / OUT | Program IN / OUT |
|---|---|---|---|
| 1 | One person, walking normally | 5 / 5 | |
| 2 | One person, walking fast | 5 / 5 | |
| 3 | Two people, one after the other | 5 / 5 | |
| 4 | Two people, side by side | 5 / 5 | |
| 5 | A person stops on the line and turns back | 0 / 0 | |

**Accuracy** = `100 - (|real - program| / real × 100)`. For example, real IN 5 and program IN 4 gives `100 - 20 = 80 %`.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 result and accuracy:
- Test 2 result and accuracy:
- Test 3 result and accuracy:
- Test 4 result and accuracy:
- Test 5 result (a person who only turns back should not be counted):
- Which situation was the hardest for the tracker and why?:
- MARGIN and CONFIDENCE values that worked best:


### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration** to your instructor: show the live view, cross the line IN and OUT, show the log in Excel and explain the numbers of your test.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Place where you would use the counter:
- Model file used and the FPS shown:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- Overall accuracy of your counter:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** A counter like this runs in shops, libraries, lifts and bus doors. In the next lab you will use the same live-view code with a different model that finds the **body points** of a person (pose estimation), so the program can tell if a person is standing, sitting or has raised a hand.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The tracker runs live and each person keeps the same ID while walking | |
| 2 | The answers about the ID tests in Part B are complete | |
| 3 | The line, the margin and `IN` / `OUT` / `INSIDE` show on the screen | |
| 4 | A person walks IN and then OUT and the counts are right | |
| 5 | `ROOM FULL` shows when the room limit is reached | |
| 6 | The log is shown in Excel (`people_log.xlsx`) with at least 4 lines | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table in Part E is complete, with the accuracy calculated | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What is the difference between detection and tracking? Give one question that a detector can answer and one that needs a tracker.:
- 2. What does `persist=True` do in `model.track(...)`? What would you see on the screen without it?:
- 3. Why does the tracker give a new ID when a person is hidden for too long? What does this do to the count?:
- 4. Explain how `check_crossing` decides that a person went IN. Why does it remember the last side of each ID?:
- 5. What is the margin for? What happens if `MARGIN = 0` and a person stands on the line?:
- 6. Why is `inside = IN - OUT` not always right? Name one situation.:
- 7. A shop owner wants the number of customers per hour. Which columns of `people_log.csv` would you use, and how would you get the answer in Excel?:
- 8. Your counter shows 8 IN when 10 people walked in. What is the accuracy? Name one change that could help.:
- 9. Which is the best place for the camera to count people at a door? (a) Beside the door at the height of the knees (b) Above the door, looking down (c) Far across the street (d) Behind a window with a reflection:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `No module named 'lap'` | Step 2 of Part A was not run, or there is no internet | Run Step 2 of Part A again. Check the internet with your teacher. |
| `yolo26n_fp16.engine not found, using yolo26n.pt instead` | The engine is not in this folder | Run Step 1 of Part A again, or build the engine in Lab 15 |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| No picture appears | The cells above were not run, or an earlier error | Run the cells above from the top. Read the error message first. |
| The ID changes all the time | `persist=True` was removed, or the confidence limit is too high so the person flickers | Check the `model.track(...)` line. Lower the confidence slider a little. |
| Nothing is counted when I cross the line | The line is not between the start and the end of your walk, or you walk along the line | Move the line with the slider. Cross it sideways. |
| One walk is counted many times | `MARGIN` is too small, or the tracker lost the person and gave a new ID | Raise `MARGIN`. Walk slower. Do not overlap with other people. |
| IN and OUT are the wrong way round | Your IN side is the other side | Set `FLIP = True`, run the settings and the program cells again |
| The numbers do not reset | The counter keeps its numbers while the kernel runs | Run the cell of Step 2 again: it starts with `reset_tracker()` |
| `CUDA out of memory` | Other notebooks and programs use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(10)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Track mode: https://docs.ultralytics.com/modes/track/
- ByteTrack paper (Zhang et al.): https://arxiv.org/abs/2110.06864
- Ultralytics, Object counting solution: https://docs.ultralytics.com/guides/object-counting/
- openpyxl, working with Excel files: https://openpyxl.readthedocs.io/